In [1]:
#This script loads your LPG infrastructure layers from the GeoPackage, opens each point in Google Maps for visual inspection, and prompts you to classify it
#(e.g., primary hub, depot, filling plant, retailer, or invalid) 
#while optionally adding notes, saving your classifications back to the file.

In [ ]:
import geopandas as gpd
import webbrowser
import pandas as pd

DATA_DIR = "dataset_big"
GPKG_PATH = f"{DATA_DIR}/lpg_infrastructure_3857.gpkg"
gpkg = GPKG_PATH
layers = ["both", "depot_only", "filling_only"]

codes = {
    "D": "primary depot",
    "F": "filling plant",
    "B": "both",
    "R": "retailer",
    "X": "invalid"
}

for layer in layers:
    print(f"\n--- Checking layer: {layer} ---")
    gdf = gpd.read_file(gpkg, layer=layer)
    
    if "verified_type" not in gdf.columns:
        gdf["verified_type"] = ""
    if "notes" not in gdf.columns:
        gdf["notes"] = ""
    
    # Find first unverified point (where verified_type is empty or NaN)
    start_idx = None
    for idx, row in gdf.iterrows():
        vt = row.get("verified_type")
        if pd.isna(vt) or vt == "":
            start_idx = idx
            break
    
    if start_idx is None:
        print(f"All points in {layer} already verified. Skipping.")
        continue
    
    total = len(gdf)
    print(f"Resuming from point {start_idx+1}/{total}")
    
    for idx in range(start_idx, total):
        row = gdf.iloc[idx]
        print(f"\n[{idx+1}/{total}] {row.get('name', 'Unnamed')}")
        print(f"Address: {row.get('address', 'N/A')}")
        print(f"Keyword: {row.get('keyword', 'N/A')}")
        lat, lng = row["lat"], row["lng"]
        url = f"https://www.google.com/maps?q={lat},{lng}"
        print(f"Maps: {url}")
        webbrowser.open(url)
        
        while True:
            choice = input("Type (D/F/B/R/X, Enter to skip): ").strip().upper()
            if choice == "":
                break
            if choice in codes:
                gdf.at[idx, "verified_type"] = codes[choice]
                break
            else:
                print("Invalid. Use D, F, B, R, X")
        
        if choice != "":
            note = input("Notes (optional): ").strip()
            gdf.at[idx, "notes"] = note
        
        if (idx + 1) % 20 == 0:
            gdf.to_file(gpkg, layer=layer, driver="GPKG")
            print(f"Checkpoint saved for layer {layer}")
    
    gdf.to_file(gpkg, layer=layer, driver="GPKG")
    print(f"Layer {layer} finished and saved.")


--- Checking layer: both ---
All points in both already verified. Skipping.

--- Checking layer: depot_only ---
All points in depot_only already verified. Skipping.

--- Checking layer: filling_only ---
All points in filling_only already verified. Skipping.


In [ ]:
import geopandas as gpd
import pandas as pd
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

gpkg = GPKG_PATH
layers = ["both", "depot_only", "filling_only"]

dfs = []
for layer in layers:
    gdf = gpd.read_file(gpkg, layer=layer)
    gdf["original_layer"] = layer
    dfs.append(gdf)

combined = pd.concat(dfs, ignore_index=True)
classified = combined[combined["verified_type"].notna()].copy()
print(f"Total classified: {len(classified)}")
print("\nClass distribution:")
print(classified["verified_type"].value_counts())

# Prepare text features (name + keyword)
classified["text"] = classified["name"].fillna("") + " " + classified["keyword"].fillna("")
X_text = classified["text"]
y = classified["verified_type"]

# TF‑IDF + Logistic Regression
vectorizer = TfidfVectorizer(max_features=500, stop_words="english")
X_tfidf = vectorizer.fit_transform(X_text)

clf = LogisticRegression(max_iter=1000)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Filter classes with at least 2 samples for CV
min_class_size = 2
valid_classes = y.value_counts()[y.value_counts() >= min_class_size].index
mask = y.isin(valid_classes).values
X_tfidf_cv = X_tfidf[mask]
y_cv = y[mask]

# Check number of rows using shape[0]
if X_tfidf_cv.shape[0] > 0:
    scores = cross_val_score(clf, X_tfidf_cv, y_cv, cv=cv, scoring="accuracy")
    print(f"\nCross‑validation accuracy (TF‑IDF + LogisticRegression): {scores.mean():.3f} ± {scores.std():.3f}")
else:
    print("\nNot enough samples for cross‑validation.")

# Train on all data to get feature importances
clf.fit(X_tfidf, y)
feature_names = np.array(vectorizer.get_feature_names_out())
coef = clf.coef_

print("\nTop 10 words for each class (highest positive coefficient):")
for i, class_name in enumerate(clf.classes_):
    top_features = feature_names[np.argsort(coef[i])[-10:]]
    print(f"\n{class_name}:")
    for f in top_features[::-1]:
        print(f"  {f}")

# Create a refined rule‑based classifier
def refine_rule(row):
    name = row["name"].lower()
    kw = row["keyword"].lower()
    orig = row["original_layer"]

    # Invalid first: installation, engineering, services (unless it's a clear gas facility)
    if any(term in name for term in ["installation", "engineering", "construction", "fabrication", "consult", "training", "services ltd"]):
        # But if name contains "gas" and also "bottling" or "filling", override
        if "gas" in name and any(term in name for term in ["bottling", "filling", "refilling", "cylinder"]):
            pass
        else:
            return "invalid"

    # Primary depot: terminal, depot, storage, and not bottling/refilling
    if any(term in name for term in ["terminal", "depot", "storage"]) and not any(term in name for term in ["bottling", "refilling", "filling", "cylinder"]):
        return "primary depot"

    # Filling plant: bottling, refilling, filling, cylinder, and not clearly retail
    if any(term in name for term in ["bottling", "refilling", "filling", "cylinder"]):
        # If name also contains "shop" or "retail", could be retailer, but often these are filling plants
        if "shop" in name or "retail" in name:
            return "retailer"
        return "filling plant"

    # Retailer: cooking gas, shop, retail, but not engineering/installation
    if any(term in name for term in ["cooking gas", "shop", "retail", "gas shop"]):
        return "retailer"

    # If still undecided, use original layer
    if orig == "both":
        # Most both points ended up as filling plants
        return "filling plant"
    elif orig == "depot_only":
        # Many depot_only points are invalid or retailer
        # But if keyword suggests depot, keep as primary depot?
        if "depot" in kw or "terminal" in kw:
            return "primary depot"
        else:
            return "invalid"  # fallback
    elif orig == "filling_only":
        # Most filling_only points are filling plants or invalid
        return "filling plant"
    else:
        return "invalid"

classified["rule_pred"] = classified.apply(refine_rule, axis=1)

print("\n=== Refined rule‑based classifier performance ===")
crosstab = pd.crosstab(classified["verified_type"], classified["rule_pred"], margins=True)
print(crosstab)

# Identify cases where refined rules still fail
mis = classified[classified["verified_type"] != classified["rule_pred"]]
print(f"\nMisclassifications with refined rules: {len(mis)} / {len(classified)}")
if len(mis) > 0:
    print("\nFirst 20 misclassifications:")
    for _, row in mis.head(20).iterrows():
        print(f"  {row['name']} | manual: {row['verified_type']} | rule: {row['rule_pred']} | kw: {row['keyword']} | orig: {row['original_layer']}")



Total classified: 670

Class distribution:
verified_type
filling plant    376
invalid          190
retailer          90
primary depot     14
Name: count, dtype: int64

Cross‑validation accuracy (TF‑IDF + LogisticRegression): 0.575 ± 0.036

Top 10 words for each class (highest positive coefficient):

filling plant:
  petrol
  rano
  gasland
  petrocam
  station
  petroleum
  road
  limited
  abound
  brim

invalid:
  cng
  rainoil
  asad
  engineering
  galadimawa
  lng
  micro
  nigeria
  hydropet
  ado

primary depot:
  terminal
  depot
  nipco
  liquid
  gasterminalling
  plc
  escravos
  distribution
  storage
  npsc

retailer:
  shop
  gas
  taraba
  better
  aa
  services
  filling
  cylinders
  eco
  gaslink

=== Refined rule‑based classifier performance ===
rule_pred      filling plant  invalid  primary depot  retailer  All
verified_type                                                      
filling plant            347        9              7        13  376
invalid              